# 01 Merge Names And Engineer Sequence Features

Build the modeling table. The pipeline prints diagnostics before and after the lineup-handedness merge.

In [ ]:
# Colab/local setup. Keep helper functions at the top of every notebook.
import os
import sys
import subprocess
from pathlib import Path

# Detect if running in Google Colab environment
IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""  # Optional: set after the public GitHub repo exists.
TOP_N = 100

# Set up the base directory based on whether it's Colab or a local environment
if IN_COLAB:
    # Install necessary packages quietly if in Colab
    %pip -q install -r requirements.txt || %pip -q install pybaseball pandas numpy scikit-learn matplotlib seaborn pyarrow shap statsmodels xgboost tqdm
    # Clone the repository if a URL is provided and it's not already cloned
    if REPO_URL and not Path("/content/pitch-sequencing").exists():
        !git clone {REPO_URL} /content/pitch-sequencing
    # Set BASE_DIR to the cloned repo path or current working directory
    BASE_DIR = Path("/content/pitch-sequencing") if Path("/content/pitch-sequencing").exists() else Path.cwd()
else:
    BASE_DIR = Path.cwd()

# Change current directory to BASE_DIR
os.chdir(BASE_DIR)
# Add the 'code' directory to the system path for module imports
sys.path.insert(0, str(BASE_DIR / "code"))

# Define paths for data and output directories
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "output"

# Create necessary directories if they don't exist
for path in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, OUTPUT_DIR / "figures"]:
    path.mkdir(parents=True, exist_ok=True)

# Helper function to run external commands and print their output
def run_step(args):
    print("Running:", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=BASE_DIR, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

# Helper function to print DataFrame diagnostics (rows, columns, unique pitchers, pitch types)
def frame_diag(df, label):
    print(f"{label}: rows={len(df):,}, cols={df.shape[1]:,}")
    if "pitcher_name" in df:
        print(f"{label}: pitchers={df['pitcher_name'].nunique():,}")
    if "pitch_type" in df:
        print(f"{label}: pitch_types={df['pitch_type'].nunique():,}")
        print(df["pitch_type"].value_counts().head(12))
    return df.head()

# Helper function to perform a left merge and print diagnostics before and after
def merge_diag(left, right, keys, label):
    print(f"[merge:{label}] before left_rows={len(left):,}, right_rows={len(right):,}, keys={keys}")
    print(f"[merge:{label}] right_duplicate_keys={right.duplicated(keys).sum():,}")
    merged = left.merge(right, on=keys, how="left", validate="many_to_one", indicator=True)
    print(f"[merge:{label}] after rows={len(merged):,}, unmatched={merged['_merge'].eq('left_only').sum():,}")
    return merged.drop(columns=["_merge"])

# Helper function to display the head of a CSV file from a given path
def show_csv(path, rows=8):
    import pandas as pd
    path = BASE_DIR / path
    print(path)
    df = pd.read_csv(path)
    display(df.head(rows))
    return df

In [ ]:
from pitch_sequence_pipeline import (
    add_player_names,
    build_sequence_features,
    cache,
    ensure_dirs,
    pull_pitcher_data,
    sequence_features_path,
)
import pandas as pd

# Ensure all necessary directories exist
ensure_dirs()
# Enable caching for data loading operations
cache.enable()

# Load the list of top N pitchers for 2025
pitchers = pd.read_csv(OUTPUT_DIR / f"top_{TOP_N}_pitchers_2025.csv")

# Pull raw Statcast data for 2025 for the specified date range and pitchers
raw_2025 = pull_pitcher_data(2025, "2025-03-18", "2025-09-28", pitchers, raw_label="statcast")
# Display diagnostics for the raw data before name merging
frame_diag(raw_2025, "raw_before_name_merge")

# Add player names to the raw data
named_2025 = add_player_names(raw_2025, pitchers)
# Display diagnostics for the data after player name mapping
frame_diag(named_2025, "named_after_player_mapping")

# Build sequence features from the named data
features_2025 = build_sequence_features(named_2025, output_path=sequence_features_path(2025, TOP_N))
# Display diagnostics for the engineered features
frame_diag(features_2025, "features_2025")
# Display the head of the features DataFrame with selected columns
display(features_2025[["game_date", "pitcher_name", "batter_name", "stand", "lineup_left_share", "count", "prev_pitch_type", "pitch_type"]].head())

Outputs: `data/processed/sequence_features_2025_top100.parquet` plus batter lookup tables.